# Tujuan

Notebook ini bertanggung jawab untuk:

1. Preparasi & Verifikasi File Sumber
    - Memvalidasi source file (existence, tipe, ukuran).
    - Mengambil fingerprint source menggunakan SHA-256.
    - Memvalidasi struktur CSV mentah (encoding, delimiter, duplicate header).
    - Membaca raw dataset.

2. Standarisasi & Validasi Struktur Data
    - Menstandarkan nama kolom ke lowercase.
    - Memvalidasi schema dan data type terhadap satu *schema contract* tunggal.
    - Memvalidasi primary key (`passengerid`).
    - Memvalidasi duplicate record (full-row **dan** business-key).

3. Validasi Kualitas & Domain Data
    - Membuat profiling missing values & memvalidasi threshold missingness.
    - Memvalidasi kualitas string (empty / whitespace) dan format nama.
    - Memvalidasi domain numerik dan domain kategorikal.
    - Memvalidasi hubungan train/test (ID overlap) dan kewajaran jumlah baris.

4. Penyimpanan & Verifikasi Artefak (Staging)
    - Menyimpan dataset ke staged Parquet.
    - Memvalidasi ulang staged artifact (read-back + schema fingerprint match).

5. Dokumentasi, Metadata & Provenance
    - Membuat validation report (JSON + Markdown) yang **selalu tertulis**, baik proses lolos maupun gagal.
    - Membuat ingestion metadata lengkap.
    - Mencatat provenance (git) dan environment.

6. Pembersihan Sistem
    - Membersihkan resource setelah proses selesai.

# Boundary

Notebook ini **tidak melakukan data cleaning substantif**.

Tidak dilakukan:

- imputasi missing value
- penghapusan duplicate
- penghapusan outlier
- encoding
- feature engineering
- perubahan nilai data
- koreksi data source

Normalisasi nama kolom ke lowercase hanya merupakan standardisasi schema teknis. Jika ditemukan masalah pada raw dataset, masalah tersebut **dideteksi dan dilaporkan**, bukan diperbaiki di notebook ini.

# Preparation & Verification

## Import & Configuration

In [1]:
from pathlib import Path
from datetime import datetime, timezone
import sys
import csv
import gc
import hashlib
import io
import json
import platform
import subprocess
import time

import polars as pl

# Polars Config
pl.Config.set_tbl_rows(-1)
pl.Config.set_tbl_cols(-1)

# Path Project
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

# print(PROJECT_ROOT)

## Path Configuration

In [2]:
# Raw location
RAW_DIR = PROJECT_ROOT / "data" / "raw"
TRAIN_RAW = RAW_DIR / "train.csv"
TEST_RAW = RAW_DIR / "test.csv"

# Staging location
STAGED_DIR = PROJECT_ROOT / "data" / "stage"

# Parquet
TRAIN_PARQUET = STAGED_DIR / "train.parquet"
TEST_PARQUET = STAGED_DIR / "test.parquet"

# Report
METADATA_DIR = STAGED_DIR / "metadata"
HISTORY_DIR = METADATA_DIR / "history"

INGESTION_METADATA = METADATA_DIR / "ingestion_metadata.json"
VALIDATION_REPORT = METADATA_DIR / "validation_report.json"

VALIDATION_REPORT_MD = METADATA_DIR / "validation_report.md"

# Dir
STAGED_DIR.mkdir(parents = True, exist_ok = True)
METADATA_DIR.mkdir(parents = True, exist_ok = True)
HISTORY_DIR.mkdir(parents = True, exist_ok = True)

print(f"TRAIN_RAW  : {TRAIN_RAW}")
print(f"TEST_RAW   : {TEST_RAW}")
print(f"STAGED_DIR : {STAGED_DIR}")
print(f"METADATA   : {METADATA_DIR}")
print(f"HISTORY    : {HISTORY_DIR}")

TRAIN_RAW  : d:\GIT DATA\titanic_survival_analysis\data\raw\train.csv
TEST_RAW   : d:\GIT DATA\titanic_survival_analysis\data\raw\test.csv
STAGED_DIR : d:\GIT DATA\titanic_survival_analysis\data\stage
METADATA   : d:\GIT DATA\titanic_survival_analysis\data\stage\metadata
HISTORY    : d:\GIT DATA\titanic_survival_analysis\data\stage\metadata\history


## Pipeline Contract

In [3]:
DATASET_NAME = "titanic"
PIPELINE_STAGE = "ingestion"
PIPELINE_VERSION = "1.0.0"
SCHEMA_VERSION = "1.0.0"
DQ_RULES_VERSION = "1.0.0"
SOURCE_FORMAT = "csv"
ARTIFACT_FORMAT = "parquet"

print(f"Dataset          : {DATASET_NAME}")
print(f"Pipeline stage   : {PIPELINE_STAGE}")
print(f"Pipeline version : {PIPELINE_VERSION}")
print(f"Schema version   : {SCHEMA_VERSION}")
print(f"DQ rules version : {DQ_RULES_VERSION}")

Dataset          : titanic
Pipeline stage   : ingestion
Pipeline version : 1.0.0
Schema version   : 1.0.0
DQ rules version : 1.0.0


## Run Identification

In [4]:
INGESTION_START = time.perf_counter()
RUN_TIMESTAMP = datetime.now(timezone.utc)
RUN_TIMESTAMP_COMPACT = RUN_TIMESTAMP.strftime("%Y%m%dT%H%M%SZ")

RUN_ID = f"{DATASET_NAME}-{PIPELINE_STAGE}-{RUN_TIMESTAMP_COMPACT}"

print(f"Run ID  : {RUN_ID}")
print(f"Started : {RUN_TIMESTAMP.isoformat()}")

Run ID  : titanic-ingestion-20260922T032421Z
Started : 2026-09-22T03:24:21.428320+00:00


## Schema Contract — Single Source of Truth

In [5]:
# Urutan kolom kanonik dataset Titanic. "survived" hanya ada di train.
COLUMN_ORDER = [
    "passengerid",
    "survived",
    "pclass",
    "name",
    "sex",
    "age",
    "sibsp",
    "parch",
    "ticket",
    "fare",
    "cabin",
    "embarked",
]

TRAIN_ONLY_COLUMNS = {"survived"}

DTYPE_NAME_MAP = {
    "Int64": pl.Int64,
    "Float64": pl.Float64,
    "String": pl.String,
}

SCHEMA_CONTRACT = {
    "passengerid": {
        "dtype": "Int64",
        "nullable": False,
        "required": True,
        "semantic_type": "primary_key",
    },
    "survived": {
        "dtype": "Int64",
        "nullable": False,
        "required": True,
        "semantic_type": "binary_target",
        "allowed_values": [0, 1],
    },
    "pclass": {
        "dtype": "Int64",
        "nullable": False,
        "required": True,
        "semantic_type": "categorical",
        "allowed_values": [1, 2, 3],
    },
    "name": {
        "dtype": "String",
        "nullable": False,
        "required": True,
        "semantic_type": "text",
    },
    "sex": {
        "dtype": "String",
        "nullable": False,
        "required": True,
        "semantic_type": "categorical",
        "allowed_values": ["male", "female"],
    },
    "age": {
        "dtype": "Float64",
        "nullable": True,
        "required": False,
        "semantic_type": "numeric",
        "min": 0,
        "max": 100,
        "max_null_ratio": 0.30,
    },
    "sibsp": {
        "dtype": "Int64",
        "nullable": False,
        "required": True,
        "semantic_type": "count",
        "min": 0,
    },
    "parch": {
        "dtype": "Int64",
        "nullable": False,
        "required": True,
        "semantic_type": "count",
        "min": 0,
    },
    "ticket": {
        "dtype": "String",
        "nullable": False,
        "required": True,
        "semantic_type": "identifier",
    },
    "fare": {
        "dtype": "Float64",
        "nullable": True,
        "required": False,
        "semantic_type": "numeric",
        "min": 0,
        "max_null_ratio": 0.01,
    },
    "cabin": {
        "dtype": "String",
        "nullable": True,
        "required": False,
        "semantic_type": "categorical_text",
        "max_null_ratio": 0.90,
    },
    "embarked": {
        "dtype": "String",
        "nullable": True,
        "required": False,
        "semantic_type": "categorical",
        "allowed_values": ["C", "Q", "S"],
        "max_null_ratio": 0.01,
    },
}

assert set(COLUMN_ORDER) == set(SCHEMA_CONTRACT), (
    "COLUMN_ORDER dan SCHEMA_CONTRACT tidak sinkron"
)

print("✓ SCHEMA_CONTRACT terdefinisi untuk", len(SCHEMA_CONTRACT), "kolom")

✓ SCHEMA_CONTRACT terdefinisi untuk 12 kolom


In [6]:
# ============================================================
# DERIVED CONTRACTS (diturunkan dari SCHEMA_CONTRACT)
# ============================================================

EXPECTED_TRAIN_COLUMNS = list(COLUMN_ORDER)

EXPECTED_TEST_COLUMNS = [
    column for column in COLUMN_ORDER
    if column not in TRAIN_ONLY_COLUMNS
]

EXPECTED_DTYPES = {
    column: DTYPE_NAME_MAP[spec["dtype"]]
    for column, spec in SCHEMA_CONTRACT.items()
}

STRING_COLUMNS = [
    column for column, spec in SCHEMA_CONTRACT.items()
    if spec["dtype"] == "String"
]

NUMERIC_BOUNDS = {
    column: {"min": spec.get("min"), "max": spec.get("max")}
    for column, spec in SCHEMA_CONTRACT.items()
    if spec["dtype"] in ("Int64", "Float64") and ("min" in spec or "max" in spec)
}

ALLOWED_VALUES = {
    column: set(spec["allowed_values"])
    for column, spec in SCHEMA_CONTRACT.items()
    if "allowed_values" in spec
}

MISSINGNESS_THRESHOLDS = {
    column: spec.get("max_null_ratio", 0.0 if not spec["nullable"] else 1.0)
    for column, spec in SCHEMA_CONTRACT.items()
}

REQUIRED_NON_NULL = {
    "train": [
        column for column in EXPECTED_TRAIN_COLUMNS
        if not SCHEMA_CONTRACT[column]["nullable"]
    ],
    "test": [
        column for column in EXPECTED_TEST_COLUMNS
        if not SCHEMA_CONTRACT[column]["nullable"]
    ],
}

print("EXPECTED_TRAIN_COLUMNS :", EXPECTED_TRAIN_COLUMNS)
print("EXPECTED_TEST_COLUMNS  :", EXPECTED_TEST_COLUMNS)
print("STRING_COLUMNS         :", STRING_COLUMNS)
print("ALLOWED_VALUES keys    :", list(ALLOWED_VALUES.keys()))
print("MISSINGNESS_THRESHOLDS :", MISSINGNESS_THRESHOLDS)
print("REQUIRED_NON_NULL[train]:", REQUIRED_NON_NULL["train"])
print("REQUIRED_NON_NULL[test] :", REQUIRED_NON_NULL["test"])


EXPECTED_TRAIN_COLUMNS : ['passengerid', 'survived', 'pclass', 'name', 'sex', 'age', 'sibsp', 'parch', 'ticket', 'fare', 'cabin', 'embarked']
EXPECTED_TEST_COLUMNS  : ['passengerid', 'pclass', 'name', 'sex', 'age', 'sibsp', 'parch', 'ticket', 'fare', 'cabin', 'embarked']
STRING_COLUMNS         : ['name', 'sex', 'ticket', 'cabin', 'embarked']
ALLOWED_VALUES keys    : ['survived', 'pclass', 'sex', 'embarked']
MISSINGNESS_THRESHOLDS : {'passengerid': 0.0, 'survived': 0.0, 'pclass': 0.0, 'name': 0.0, 'sex': 0.0, 'age': 0.3, 'sibsp': 0.0, 'parch': 0.0, 'ticket': 0.0, 'fare': 0.01, 'cabin': 0.9, 'embarked': 0.01}
REQUIRED_NON_NULL[train]: ['passengerid', 'survived', 'pclass', 'name', 'sex', 'sibsp', 'parch', 'ticket']
REQUIRED_NON_NULL[test] : ['passengerid', 'pclass', 'name', 'sex', 'sibsp', 'parch', 'ticket']


Threshold missingness di atas adalah **policy untuk dataset Titanic project ini**,
bukan aturan universal untuk semua dataset. Nilainya sengaja disematkan pada
`SCHEMA_CONTRACT` (bukan dipisah ke dict independen) agar setiap perubahan
kontrak kolom otomatis konsisten dengan threshold-nya.

In [7]:
# ============================================================
# MISSING VALUE TOKENS
# ============================================================

# Hanya token yang memang dianggap missing oleh source contract.
#
# Jangan memasukkan token generik seperti "No" secara global
# karena dapat merupakan nilai valid pada dataset lain.

MISSING_VALUES = [
    "",
    "NA",
    "N/A",
    "na",
    "n/a",
    "N/a",
]


In [8]:
# ============================================================
# REFERENCE BASELINE (drift check, non-blocking)
# ============================================================

# Dataset kompetisi Titanic (Kaggle) bersifat statis/final: 891 baris train,
# 418 baris test. Baseline ini HANYA dipakai sebagai sinyal WARNING untuk
# menangkap kesalahan operasional (mis. file salah / ter-corrupt / ter-subset
# tanpa sengaja), bukan sebagai aturan struktural yang blocking — karena
# turunan/subset dataset yang sah tetap mungkin dipakai pada eksperimen lain.

REFERENCE_ROW_COUNTS = {
    "train": 891,
    "test": 418,
}

ROW_COUNT_DRIFT_TOLERANCE = 0  # dalam jumlah baris; 0 = harus sama persis


In [9]:
# ============================================================
# SHA-256
# ============================================================

def sha256_file(path: Path, chunk_size: int = 1024 * 1024) -> str:
    sha256 = hashlib.sha256()

    with path.open("rb") as file:
        for chunk in iter(lambda: file.read(chunk_size), b""):
            sha256.update(chunk)

    return sha256.hexdigest()


# Validation Rule Engine

In [10]:
# ============================================================
# VALIDATION RESULT STORE
# ============================================================

validation_results = []


class IngestionBlockingError(RuntimeError):
    # Dilempar ketika sebuah rule blocking gagal / error.
    pass


def _serialize_value(value):
    if value is None:
        return None

    if isinstance(value, (dict, list, tuple, set)):
        if isinstance(value, set):
            value = sorted(value)
        elif isinstance(value, tuple):
            value = list(value)

        return json.dumps(value, ensure_ascii=False, sort_keys=True, default=str)

    return str(value)


def record(rule_id, check, dataset, status, severity, actual=None, expected=None, message=""):
    # Menambahkan satu hasil validasi dengan struktur konsisten.

    entry = {
        "rule_id": str(rule_id),
        "check": str(check),
        "dataset": str(dataset),
        "status": str(status),
        "severity": str(severity),
        "actual": _serialize_value(actual),
        "expected": _serialize_value(expected),
        "message": str(message),
    }

    validation_results.append(entry)

    marker = {"PASS": "✓", 
              "WARNING": "⚠", 
              "FAIL": "✗", 
              "ERROR": "‼"}.get(status, "?")
    
    print(f"{marker} [{rule_id}] {check} ({dataset}) -> {status}: {message}")

    return entry


def write_partial_report_on_blocking_failure(rule_id, message):
    # Menulis jejak audit sebisa mungkin sebelum notebook dihentikan oleh
    # kegagalan blocking. Defensif terhadap variabel yang belum ada di globals().

    partial = {
        "run_id": globals().get("RUN_ID", "unknown"),
        "dataset": globals().get("DATASET_NAME", "unknown"),
        "pipeline_stage": globals().get("PIPELINE_STAGE", "ingestion"),
        "timestamp_utc": globals().get("RUN_TIMESTAMP", datetime.now(timezone.utc)).isoformat()
            if hasattr(globals().get("RUN_TIMESTAMP", None), "isoformat")
            else datetime.now(timezone.utc).isoformat(),
        "overall_status": "FAIL",
        "blocking_rule_id": rule_id,
        "blocking_message": message,
        "rules_recorded_before_failure": validation_results,
    }

    failure_path = HISTORY_DIR / f"{globals().get('RUN_ID', 'unknown')}_BLOCKING_FAILURE.json"

    try:
        failure_path.write_text(
            json.dumps(partial, indent=2, ensure_ascii=False),
            encoding="utf-8",
        )
        print(f"‼ Partial audit trail written to: {failure_path}")
    except Exception as exc:
        print(f"‼ Gagal menulis partial audit trail: {exc}")


def run_rule(rule_id, check, dataset, severity, blocking, func, *args, **kwargs):
    # Menjalankan satu fungsi pemeriksaan `func(*args, **kwargs)` yang WAJIB
    # mengembalikan dict {"status", "actual", "expected", "message"}.
    #
    # Exception tak terduga ditangkap dan dicatat sebagai status "ERROR"
    # (bukan meng-crash notebook secara diam-diam).

    try:
        result = func(*args, **kwargs)
    except Exception as exc:
        result = {
            "status": "ERROR",
            "actual": repr(exc),
            "expected": "successful execution",
            "message": f"Unexpected error while running {check}: {exc}",
        }

    entry = record(
        rule_id=rule_id,
        check=check,
        dataset=dataset,
        status=result["status"],
        severity=severity,
        actual=result.get("actual"),
        expected=result.get("expected"),
        message=result.get("message", ""),
    )

    if blocking and result["status"] in ("FAIL", "ERROR"):
        write_partial_report_on_blocking_failure(rule_id, result.get("message", ""))
        raise IngestionBlockingError(
            f"[{rule_id}] {check}: BLOCKING FAILURE — {result.get('message', '')}"
        )

    return entry


### DQ-001 — Source File Validation *(blocking)*

In [11]:
def check_source_files() -> dict:
    info = {}

    for name, path in (("train", TRAIN_RAW), ("test", TEST_RAW)):
        if not path.exists():
            return {
                "status": "FAIL",
                "actual": f"{path} does not exist",
                "expected": "file exists",
                "message": f"Source file tidak ditemukan: {path}",
            }

        if not path.is_file():
            return {
                "status": "FAIL",
                "actual": f"{path} is not a regular file",
                "expected": "regular file",
                "message": f"Path bukan file: {path}",
            }

        if path.suffix.lower() != ".csv":
            return {
                "status": "FAIL",
                "actual": path.suffix.lower(),
                "expected": ".csv",
                "message": f"Source file harus CSV: {path}",
            }

        size_bytes = path.stat().st_size

        if size_bytes <= 0:
            return {
                "status": "FAIL",
                "actual": 0,
                "expected": "> 0 bytes",
                "message": f"Source file kosong: {path}",
            }

        info[name] = {
            "path": str(path),
            "file_name": path.name,
            "extension": path.suffix.lower(),
            "size_bytes": size_bytes,
        }

    return {
        "status": "PASS",
        "actual": info,
        "expected": "existing, non-empty .csv files",
        "message": "Source files exist and are valid CSV files.",
    }


file_check = run_rule(
    "DQ-001", "source_files", "all", "CRITICAL", True,
    check_source_files,
)

train_file_info = json.loads(file_check["actual"])["train"]
test_file_info = json.loads(file_check["actual"])["test"]


✓ [DQ-001] source_files (all) -> PASS: Source files exist and are valid CSV files.


### DQ-002 — Source Fingerprint (SHA-256) *(non-blocking, informational)*

In [12]:
def check_source_fingerprint() -> dict:
    train_hash = sha256_file(TRAIN_RAW)
    test_hash = sha256_file(TEST_RAW)

    return {
        "status": "PASS",
        "actual": {"train_sha256": train_hash, "test_sha256": test_hash},
        "expected": "valid SHA-256 fingerprint",
        "message": "Source SHA-256 fingerprints generated successfully.",
    }


fingerprint_check = run_rule(
    "DQ-002", "source_fingerprint", "all", "CRITICAL", False,
    check_source_fingerprint,
)

_fp = json.loads(fingerprint_check["actual"])
train_sha256 = _fp["train_sha256"]
test_sha256 = _fp["test_sha256"]

print()
print("Train SHA256:", train_sha256)
print("Test  SHA256:", test_sha256)


✓ [DQ-002] source_fingerprint (all) -> PASS: Source SHA-256 fingerprints generated successfully.

Train SHA256: 7d118fef8b6ccf7f81111877bc388536f7b1e498a655e3d649d19aaa010e9f6f
Test  SHA256: 56023b9948236f3c7a1c9448fcf418b283e109ef177fa8c7e069158dd7dd52b2


### DQ-003 — CSV Structural Validation *(blocking)*

Menggunakan `csv.reader` (quote-aware) untuk parsing header, bukan
`str.split(",")` naif, agar tetap benar walau ada nama kolom yang memuat koma
di dalam tanda kutip.

In [13]:
def check_csv_structure(path: Path) -> dict:
    raw_bytes = path.read_bytes()

    if len(raw_bytes) == 0:
        return {
            "status": "FAIL",
            "actual": 0,
            "expected": "> 0 bytes",
            "message": f"CSV kosong: {path}",
        }

    try:
        text = raw_bytes.decode("utf-8-sig")
    except UnicodeDecodeError as exc:
        return {
            "status": "FAIL",
            "actual": str(exc),
            "expected": "utf-8 (or utf-8-sig) decodable",
            "message": f"{path} bukan UTF-8 compatible CSV: {exc}",
        }

    lines = text.splitlines()

    if len(lines) < 2:
        return {
            "status": "FAIL",
            "actual": len(lines),
            "expected": ">= 2 lines",
            "message": f"{path} harus memiliki header dan minimal satu data row",
        }

    header_line = lines[0]

    if "," not in header_line:
        return {
            "status": "FAIL",
            "actual": header_line,
            "expected": "comma-delimited header",
            "message": f"{path} tidak terlihat menggunakan delimiter ','",
        }

    # Quote-aware header parsing (menangani kolom bertanda kutip yang
    # memuat koma di dalamnya).
    header_columns = [
        column.strip().lower()
        for column in next(csv.reader(io.StringIO(header_line)))
    ]

    duplicate_columns = sorted({
        column for column in header_columns
        if header_columns.count(column) > 1
    })

    if duplicate_columns:
        return {
            "status": "FAIL",
            "actual": duplicate_columns,
            "expected": "no duplicate header columns",
            "message": f"{path} memiliki duplicate header: {duplicate_columns}",
        }

    return {
        "status": "PASS",
        "actual": {
            "encoding": "utf-8",
            "delimiter": ",",
            "line_count": len(lines),
            "header": header_columns,
        },
        "expected": "utf-8 CSV, comma-delimited, unique headers",
        "message": "CSV structural validation passed.",
    }


train_csv_check = run_rule(
    "DQ-003", "csv_structure", "train", "CRITICAL", True,
    check_csv_structure, TRAIN_RAW,
)

test_csv_check = run_rule(
    "DQ-003", "csv_structure", "test", "CRITICAL", True,
    check_csv_structure, TEST_RAW,
)

train_csv_structure = json.loads(train_csv_check["actual"])
test_csv_structure = json.loads(test_csv_check["actual"])


✓ [DQ-003] csv_structure (train) -> PASS: CSV structural validation passed.
✓ [DQ-003] csv_structure (test) -> PASS: CSV structural validation passed.


### DQ-004 — Raw Data Loading *(blocking)*

In [14]:
def load_raw_csv(path: Path) -> pl.DataFrame:
    return pl.read_csv(
        path,
        null_values=MISSING_VALUES,
        try_parse_dates=False,
        infer_schema_length=1000,
    )


def check_raw_load() -> dict:
    global train, test

    try:
        train = load_raw_csv(TRAIN_RAW)
        test = load_raw_csv(TEST_RAW)
    except Exception as exc:
        return {
            "status": "FAIL",
            "actual": repr(exc),
            "expected": "CSV parses without error",
            "message": (
                f"Gagal membaca CSV. Periksa encoding, delimiter, quoting, "
                f"atau struktur CSV. Detail: {exc}"
            ),
        }

    return {
        "status": "PASS",
        "actual": {"train_shape": train.shape, "test_shape": test.shape},
        "expected": "non-empty dataframes",
        "message": "Raw CSV loaded successfully.",
    }


raw_load_check = run_rule(
    "DQ-004", "raw_load", "all", "CRITICAL", True,
    check_raw_load,
)

print("Train shape:", train.shape)
print("Test shape :", test.shape)


✓ [DQ-004] raw_load (all) -> PASS: Raw CSV loaded successfully.
Train shape: (891, 12)
Test shape : (418, 11)


### DQ-005 — Technical Column Name Standardization *(blocking)*

Normalisasi ke lowercase adalah standardisasi teknis, bukan transformasi data.

In [15]:
def normalize_column_names(df: pl.DataFrame) -> pl.DataFrame:
    normalized = [column.strip().lower() for column in df.columns]

    if len(normalized) != len(set(normalized)):
        duplicates = sorted({
            column for column in normalized
            if normalized.count(column) > 1
        })
        raise ValueError(f"Duplicate column names setelah normalisasi: {duplicates}")

    return df.rename(dict(zip(df.columns, normalized)))


def check_column_normalization() -> dict:
    global train, test

    try:
        train = normalize_column_names(train)
        test = normalize_column_names(test)
    except ValueError as exc:
        return {
            "status": "FAIL",
            "actual": str(exc),
            "expected": "no duplicate columns after lowercasing",
            "message": str(exc),
        }

    return {
        "status": "PASS",
        "actual": {"train_columns": train.columns, "test_columns": test.columns},
        "expected": "unique lowercase column names",
        "message": "Column names normalized to lowercase.",
    }


normalize_check = run_rule(
    "DQ-005", "column_normalization", "all", "CRITICAL", True,
    check_column_normalization,
)

print("Train columns:", train.columns)
print("Test columns :", test.columns)


✓ [DQ-005] column_normalization (all) -> PASS: Column names normalized to lowercase.
Train columns: ['passengerid', 'survived', 'pclass', 'name', 'sex', 'age', 'sibsp', 'parch', 'ticket', 'fare', 'cabin', 'embarked']
Test columns : ['passengerid', 'pclass', 'name', 'sex', 'age', 'sibsp', 'parch', 'ticket', 'fare', 'cabin', 'embarked']


### DQ-006 / DQ-007 — Column Schema Validation *(blocking)*

In [16]:
def check_columns(df: pl.DataFrame, expected_columns: list, dataset_name: str) -> dict:
    if len(df.columns) != len(set(df.columns)):
        return {
            "status": "FAIL",
            "actual": df.columns,
            "expected": "unique column names",
            "message": f"{dataset_name}: duplicate column names",
        }

    if df.columns != expected_columns:
        return {
            "status": "FAIL",
            "actual": df.columns,
            "expected": expected_columns,
            "message": f"{dataset_name}: column schema tidak sesuai kontrak.",
        }

    return {
        "status": "PASS",
        "actual": df.columns,
        "expected": expected_columns,
        "message": f"{dataset_name} column schema matches the contract.",
    }


train_schema_check = run_rule(
    "DQ-006", "column_schema", "train", "CRITICAL", True,
    check_columns, train, EXPECTED_TRAIN_COLUMNS, "train",
)

test_schema_check = run_rule(
    "DQ-007", "column_schema", "test", "CRITICAL", True,
    check_columns, test, EXPECTED_TEST_COLUMNS, "test",
)


✓ [DQ-006] column_schema (train) -> PASS: train column schema matches the contract.
✓ [DQ-007] column_schema (test) -> PASS: test column schema matches the contract.


### Schema Fingerprint Helper

In [17]:
def schema_fingerprint(df: pl.DataFrame) -> str:
    schema_definition = [
        {"name": column, "dtype": str(dtype)}
        for column, dtype in df.schema.items()
    ]

    canonical_schema = json.dumps(schema_definition, sort_keys=True, separators=(",", ":"))

    return hashlib.sha256(canonical_schema.encode("utf-8")).hexdigest()


train_schema_sha256 = schema_fingerprint(train)
test_schema_sha256 = schema_fingerprint(test)

print("Train schema SHA256:", train_schema_sha256)
print("Test  schema SHA256:", test_schema_sha256)


Train schema SHA256: 0f3eb72d2000aa4f1d222676c7802f36bb4a945c72879844bbf02df1ee8530d2
Test  schema SHA256: 1e26a7abd624b4491dd13e8a5993e76b0fbab542a1355e664077f7f48d7eef16


### DQ-008 — Data Type Validation *(blocking)*

Jika dtype salah, hampir semua pemeriksaan berikutnya (domain numerik,
missingness, dsb.) tidak bisa dipercaya hasilnya — karena itu rule ini tetap
`blocking=True`.

In [18]:
def check_dtypes(df: pl.DataFrame, dataset_name: str) -> dict:
    mismatches = {}

    for column, expected_dtype in EXPECTED_DTYPES.items():
        if column not in df.columns:
            continue

        actual_dtype = df.schema[column]

        if actual_dtype != expected_dtype:
            mismatches[column] = {"expected": str(expected_dtype), "actual": str(actual_dtype)}

    if mismatches:
        return {
            "status": "FAIL",
            "actual": mismatches,
            "expected": "all dtypes match SCHEMA_CONTRACT",
            "message": f"{dataset_name}: {len(mismatches)} kolom memiliki dtype tidak sesuai.",
        }

    return {
        "status": "PASS",
        "actual": {c: str(t) for c, t in df.schema.items()},
        "expected": {c: str(t) for c, t in EXPECTED_DTYPES.items()},
        "message": f"{dataset_name}: data types match the schema contract.",
    }


train_dtype_check = run_rule(
    "DQ-008", "data_types", "train", "CRITICAL", True,
    check_dtypes, train, "train",
)

test_dtype_check = run_rule(
    "DQ-008", "data_types", "test", "CRITICAL", True,
    check_dtypes, test, "test",
)


✓ [DQ-008] data_types (train) -> PASS: train: data types match the schema contract.
✓ [DQ-008] data_types (test) -> PASS: test: data types match the schema contract.


### DQ-009 — Row Count Sanity *(blocking)*

In [19]:
def check_nonzero_rows(df: pl.DataFrame, dataset_name: str) -> dict:
    if df.height <= 0:
        return {
            "status": "FAIL",
            "actual": df.height,
            "expected": "> 0",
            "message": f"{dataset_name} dataset kosong",
        }

    return {
        "status": "PASS",
        "actual": df.height,
        "expected": "> 0",
        "message": f"{dataset_name} has {df.height} rows.",
    }


run_rule("DQ-009", "row_count_nonzero", "train", "CRITICAL", True, check_nonzero_rows, train, "train")
run_rule("DQ-009", "row_count_nonzero", "test", "CRITICAL", True, check_nonzero_rows, test, "test")

✓ [DQ-009] row_count_nonzero (train) -> PASS: train has 891 rows.
✓ [DQ-009] row_count_nonzero (test) -> PASS: test has 418 rows.


{'rule_id': 'DQ-009',
 'check': 'row_count_nonzero',
 'dataset': 'test',
 'status': 'PASS',
 'severity': 'CRITICAL',
 'actual': '418',
 'expected': '> 0',
 'message': 'test has 418 rows.'}

### DQ-010 — Row Count Baseline Drift *(non-blocking)*

In [20]:
def check_row_count_baseline(df: pl.DataFrame, dataset_name: str) -> dict:
    expected = REFERENCE_ROW_COUNTS.get(dataset_name)

    if expected is None:
        return {
            "status": "PASS",
            "actual": df.height,
            "expected": "no baseline configured",
            "message": f"Tidak ada baseline row count untuk {dataset_name}, dilewati.",
        }

    drift = abs(df.height - expected)

    if drift > ROW_COUNT_DRIFT_TOLERANCE:
        return {
            "status": "WARNING",
            "actual": df.height,
            "expected": expected,
            "message": (
                f"{dataset_name}: jumlah baris ({df.height}) berbeda dari baseline "
                f"referensi Titanic ({expected}). Periksa apakah file benar/tidak ter-subset."
            ),
        }

    return {
        "status": "PASS",
        "actual": df.height,
        "expected": expected,
        "message": f"{dataset_name} row count matches the known reference baseline.",
    }


run_rule("DQ-010", "row_count_baseline", "train", "WARNING", False, check_row_count_baseline, train, "train")
run_rule("DQ-010", "row_count_baseline", "test", "WARNING", False, check_row_count_baseline, test, "test")

✓ [DQ-010] row_count_baseline (train) -> PASS: train row count matches the known reference baseline.
✓ [DQ-010] row_count_baseline (test) -> PASS: test row count matches the known reference baseline.


{'rule_id': 'DQ-010',
 'check': 'row_count_baseline',
 'dataset': 'test',
 'status': 'PASS',
 'severity': 'WARNING',
 'actual': '418',
 'expected': '418',
 'message': 'test row count matches the known reference baseline.'}

### DQ-011 — Primary Key Validation *(non-blocking)*

In [21]:
def check_primary_key(df: pl.DataFrame, dataset_name: str) -> dict:
    column = "passengerid"
    issues = []

    null_count = df[column].null_count()
    if null_count > 0:
        issues.append(f"{null_count} null value(s)")

    unique_count = df[column].n_unique()
    if unique_count != df.height:
        issues.append(f"{df.height - unique_count} duplicate identifier(s)")

    min_value = df[column].min()
    if min_value is None or min_value <= 0:
        issues.append(f"non-positive min value ({min_value})")

    if issues:
        return {
            "status": "FAIL",
            "actual": issues,
            "expected": "unique, non-null, positive passengerid",
            "message": f"{dataset_name}.{column}: " + "; ".join(issues),
        }

    return {
        "status": "PASS",
        "actual": {"null_count": 0, "unique": True, "min_value": min_value},
        "expected": "unique, non-null, positive passengerid",
        "message": f"{dataset_name}.{column} primary key validation passed.",
    }


run_rule("DQ-011", "primary_key", "train", "CRITICAL", False, check_primary_key, train, "train")
run_rule("DQ-011", "primary_key", "test", "CRITICAL", False, check_primary_key, test, "test")


✓ [DQ-011] primary_key (train) -> PASS: train.passengerid primary key validation passed.
✓ [DQ-011] primary_key (test) -> PASS: test.passengerid primary key validation passed.


{'rule_id': 'DQ-011',
 'check': 'primary_key',
 'dataset': 'test',
 'status': 'PASS',
 'severity': 'CRITICAL',
 'actual': '{"min_value": 892, "null_count": 0, "unique": true}',
 'expected': 'unique, non-null, positive passengerid',
 'message': 'test.passengerid primary key validation passed.'}

### DQ-012 — Full-Row Duplicate

Catatan penting: karena `passengerid` sudah divalidasi unik (DQ-011), cek
**full-row duplicate** secara matematis tidak akan pernah menemukan apa pun
(dua baris tidak mungkin identik jika salah satu kolomnya — `passengerid` —
pasti berbeda). Cek ini tetap dipertahankan untuk kelengkapan defensif, tetapi
**cek yang benar-benar berguna** untuk menangkap duplikasi entri manusia
(mis. penumpang yang sama ter-input dua kali dengan ID berbeda) adalah cek
*business-key* pada kombinasi `name` + `ticket`, yang ditambahkan sebagai
DQ-013.

In [22]:
def check_full_row_duplicates(df: pl.DataFrame, dataset_name: str) -> dict:
    duplicate_count = int(df.is_duplicated().sum())

    if duplicate_count > 0:
        return {
            "status": "FAIL",
            "actual": duplicate_count,
            "expected": 0,
            "message": f"{dataset_name} memiliki {duplicate_count} full-row duplicate.",
        }

    return {
        "status": "PASS",
        "actual": 0,
        "expected": 0,
        "message": f"{dataset_name}: no full-row duplicates detected.",
    }


train_dup_result = run_rule(
    "DQ-012", "full_row_duplicates", "train", "CRITICAL", False,
    check_full_row_duplicates, train, "train",
)
test_dup_result = run_rule(
    "DQ-012", "full_row_duplicates", "test", "CRITICAL", False,
    check_full_row_duplicates, test, "test",
)

train_duplicate_rows = int(json.loads(train_dup_result["actual"])) if isinstance(train_dup_result["actual"], str) else train_dup_result["actual"]
test_duplicate_rows = int(json.loads(test_dup_result["actual"])) if isinstance(test_dup_result["actual"], str) else test_dup_result["actual"]


✓ [DQ-012] full_row_duplicates (train) -> PASS: train: no full-row duplicates detected.
✓ [DQ-012] full_row_duplicates (test) -> PASS: test: no full-row duplicates detected.


### DQ-013 — Business-Key Duplicate *(non-blocking)*

In [23]:
def check_business_key_duplicates(df: pl.DataFrame, dataset_name: str, keys=("name", "ticket")) -> dict:
    available_keys = [k for k in keys if k in df.columns]

    if not available_keys:
        return {
            "status": "PASS",
            "actual": "no business key columns available",
            "expected": "n/a",
            "message": f"{dataset_name}: business-key check dilewati (kolom tidak tersedia).",
        }

    grouped = (
        df.group_by(available_keys)
        .agg(pl.len().alias("occurrences"), pl.col("passengerid").alias("passengerids"))
        .filter(pl.col("occurrences") > 1)
    )

    if grouped.height > 0:
        sample = grouped.head(10).to_dicts()
        return {
            "status": "WARNING",
            "actual": {"duplicate_groups": grouped.height, "sample": sample},
            "expected": "each (name, ticket) combination appears once",
            "message": (
                f"{dataset_name}: ditemukan {grouped.height} kombinasi "
                f"{available_keys} yang muncul lebih dari sekali (indikasi "
                f"entri ganda penumpang yang sama dengan passengerid berbeda)."
            ),
        }

    return {
        "status": "PASS",
        "actual": 0,
        "expected": 0,
        "message": f"{dataset_name}: no business-key ({available_keys}) duplicates detected.",
    }


run_rule("DQ-013", "business_key_duplicates", "train", "WARNING", False, check_business_key_duplicates, train, "train")
run_rule("DQ-013", "business_key_duplicates", "test", "WARNING", False, check_business_key_duplicates, test, "test")

print()
print("Train full-row duplicates:", train_duplicate_rows)
print("Test  full-row duplicates:", test_duplicate_rows)

✓ [DQ-013] business_key_duplicates (train) -> PASS: train: no business-key (['name', 'ticket']) duplicates detected.
✓ [DQ-013] business_key_duplicates (test) -> PASS: test: no business-key (['name', 'ticket']) duplicates detected.

Train full-row duplicates: 0
Test  full-row duplicates: 0


### Missing Value Profiling (informational display, single-pass)

In [24]:
def missing_report(df: pl.DataFrame) -> pl.DataFrame:
    if df.height == 0:
        return pl.DataFrame({
            "column": df.columns,
            "null_count": [0] * df.width,
            "null_ratio": [0.0] * df.width,
            "null_percent": [0.0] * df.width,
        })

    return (
        df.null_count()
        .transpose(include_header=True, header_name="column", column_names=["null_count"])
        .with_columns(
            (pl.col("null_count") / df.height).alias("null_ratio"),
            (pl.col("null_count") / df.height * 100).round(2).alias("null_percent"),
        )
    )


train_missing = missing_report(train)
test_missing = missing_report(test)

print("TRAIN missing value profile")
display(train_missing)

print("TEST missing value profile")
display(test_missing)


TRAIN missing value profile


column,null_count,null_ratio,null_percent
str,u32,f64,f64
"""passengerid""",0,0.0,0.0
"""survived""",0,0.0,0.0
"""pclass""",0,0.0,0.0
"""name""",0,0.0,0.0
"""sex""",0,0.0,0.0
"""age""",177,0.198653,19.87
"""sibsp""",0,0.0,0.0
"""parch""",0,0.0,0.0
"""ticket""",0,0.0,0.0


TEST missing value profile


column,null_count,null_ratio,null_percent
str,u32,f64,f64
"""passengerid""",0,0.0,0.0
"""pclass""",0,0.0,0.0
"""name""",0,0.0,0.0
"""sex""",0,0.0,0.0
"""age""",86,0.205742,20.57
"""sibsp""",0,0.0,0.0
"""parch""",0,0.0,0.0
"""ticket""",0,0.0,0.0
"""fare""",1,0.002392,0.24


### DQ-014 — Required Non-Null

`validate_missingness_threshold` pada versi sebelumnya memanggil
`df[column].null_count()` per kolom secara terpisah, padahal `missing_report()`
di atas **sudah menghitung seluruh null_count dalam satu pass**. Versi ini
menggunakan kembali `train_missing` / `test_missing` alih-alih menghitung ulang
(menghindari *redundant full-table scan*).

In [25]:
def check_required_non_null(df: pl.DataFrame, dataset_name: str) -> dict:
    required_columns = REQUIRED_NON_NULL[dataset_name]

    null_counts = df.select([
        pl.col(c).null_count().alias(c) for c in required_columns
    ]).to_dicts()[0]

    violations = {c: n for c, n in null_counts.items() if n > 0}

    if violations:
        return {
            "status": "FAIL",
            "actual": violations,
            "expected": "0 nulls in required columns",
            "message": f"{dataset_name}: kolom wajib memiliki null: {violations}",
        }

    return {
        "status": "PASS",
        "actual": {c: 0 for c in required_columns},
        "expected": "0 nulls in required columns",
        "message": f"{dataset_name}: required non-null validation passed.",
    }


run_rule("DQ-014", "required_non_null", "train", "CRITICAL", False, check_required_non_null, train, "train")
run_rule("DQ-014", "required_non_null", "test", "CRITICAL", False, check_required_non_null, test, "test")

✓ [DQ-014] required_non_null (train) -> PASS: train: required non-null validation passed.
✓ [DQ-014] required_non_null (test) -> PASS: test: required non-null validation passed.


{'rule_id': 'DQ-014',
 'check': 'required_non_null',
 'dataset': 'test',
 'status': 'PASS',
 'severity': 'CRITICAL',
 'actual': '{"name": 0, "parch": 0, "passengerid": 0, "pclass": 0, "sex": 0, "sibsp": 0, "ticket": 0}',
 'expected': '0 nulls in required columns',
 'message': 'test: required non-null validation passed.'}

### DQ-015 — Missingness Threshold *(non-blocking)*

In [26]:
def check_missingness_threshold(missing_df: pl.DataFrame, dataset_name: str) -> dict:
    violations = []

    for row in missing_df.to_dicts():
        column = row["column"]
        threshold = MISSINGNESS_THRESHOLDS.get(column)

        if threshold is None:
            continue

        if row["null_ratio"] > threshold:
            violations.append({
                "column": column,
                "null_ratio": round(row["null_ratio"], 4),
                "threshold": threshold,
            })

    if violations:
        return {
            "status": "FAIL",
            "actual": violations,
            "expected": "null_ratio <= configured threshold per column",
            "message": f"{dataset_name}: {len(violations)} kolom melebihi threshold missingness.",
        }

    return {
        "status": "PASS",
        "actual": "within configured thresholds",
        "expected": "within configured thresholds",
        "message": f"{dataset_name}: missingness is within the configured project thresholds.",
    }


run_rule("DQ-015", "missingness_threshold", "train", "CRITICAL", False, check_missingness_threshold, train_missing, "train")
run_rule("DQ-015", "missingness_threshold", "test", "CRITICAL", False, check_missingness_threshold, test_missing, "test")

✓ [DQ-015] missingness_threshold (train) -> PASS: train: missingness is within the configured project thresholds.
✓ [DQ-015] missingness_threshold (test) -> PASS: test: missingness is within the configured project thresholds.


{'rule_id': 'DQ-015',
 'check': 'missingness_threshold',
 'dataset': 'test',
 'status': 'PASS',
 'severity': 'CRITICAL',
 'actual': 'within configured thresholds',
 'expected': 'within configured thresholds',
 'message': 'test: missingness is within the configured project thresholds.'}

### DQ-016 — String Quality

Semua kolom string diperiksa dalam **satu ekspresi `.select()`** (single pass),
bukan `2 × len(STRING_COLUMNS)` pemanggilan `.filter().height` terpisah seperti
pada versi sebelumnya.

In [27]:
def check_string_quality(df: pl.DataFrame, dataset_name: str) -> dict:
    columns = [c for c in STRING_COLUMNS if c in df.columns]

    if not columns:
        return {"status": "PASS", "actual": {}, "expected": "n/a", "message": "No string columns to check."}

    exprs = []
    for c in columns:
        non_null = pl.col(c).is_not_null()
        exprs.append((non_null & (pl.col(c).str.len_chars() == 0)).sum().alias(f"{c}__empty"))
        exprs.append((non_null & (pl.col(c) != pl.col(c).str.strip_chars())).sum().alias(f"{c}__whitespace"))

    stats = df.select(exprs).to_dicts()[0]

    per_column = {}
    total_empty = 0
    total_whitespace = 0

    for c in columns:
        empty_count = int(stats[f"{c}__empty"])
        whitespace_count = int(stats[f"{c}__whitespace"])
        total_empty += empty_count
        total_whitespace += whitespace_count

        if empty_count > 0:
            status = "FAIL"
        elif whitespace_count > 0:
            status = "WARNING"
        else:
            status = "PASS"

        per_column[c] = {"empty_count": empty_count, "whitespace_count": whitespace_count, "status": status}

    if total_empty > 0:
        overall = "FAIL"
    elif total_whitespace > 0:
        overall = "WARNING"
    else:
        overall = "PASS"

    return {
        "status": overall,
        "actual": per_column,
        "expected": {"empty_strings": 0, "leading_trailing_whitespace": 0},
        "message": (
            f"{dataset_name}: string quality checked without modifying raw values "
            f"({total_empty} empty, {total_whitespace} whitespace issues)."
        ),
    }


train_string_quality_result = run_rule(
    "DQ-016", "string_quality", "train", "WARNING", False,
    check_string_quality, train, "train",
)
test_string_quality_result = run_rule(
    "DQ-016", "string_quality", "test", "WARNING", False,
    check_string_quality, test, "test",
)


def show_whitespace_issues(df: pl.DataFrame, dataset_name: str, column: str, limit: int = 20):
    if column not in df.columns:
        return

    issues = df.filter(
        pl.col(column).is_not_null() & (pl.col(column) != pl.col(column).str.strip_chars())
    )

    if issues.height == 0:
        return

    print(f"⚠ {dataset_name}.{column}: {issues.height} row memiliki leading/trailing whitespace")
    display(issues.select(["passengerid", column]).head(limit))


for _col in STRING_COLUMNS:
    show_whitespace_issues(train, "train", _col)

for _col in STRING_COLUMNS:
    show_whitespace_issues(test, "test", _col)


⚠ [DQ-016] string_quality (train) -> WARNING: train: string quality checked without modifying raw values (0 empty, 2 whitespace issues).
⚠ [DQ-016] string_quality (test) -> WARNING: test: string quality checked without modifying raw values (0 empty, 2 whitespace issues).
⚠ train.name: 2 row memiliki leading/trailing whitespace


passengerid,name
i64,str
16,"""Hewlett, Mrs. (Mary D Kingcome…"
858,"""Daly, Mr. Peter Denis """


⚠ test.name: 2 row memiliki leading/trailing whitespace


passengerid,name
i64,str
1167,"""Bryhl, Miss. Dagmar Jenny Inge…"
1222,"""Davies, Mrs. John Morgan (Eliz…"


### DQ-017 — Name Format *(non-blocking)*

In [28]:
def check_name_format(df: pl.DataFrame, dataset_name: str) -> dict:
    # Nama pada dataset Titanic mengikuti pola "Last, Title. First...".
    # Ini murni pemeriksaan sinyal kualitas data (deteksi), BUKAN parsing/
    # feature engineering — tidak ada kolom baru yang dibuat di sini.

    if "name" not in df.columns:
        return {"status": "PASS", "actual": "n/a", "expected": "n/a", "message": "Kolom name tidak tersedia."}

    malformed = df.filter(~pl.col("name").str.contains(","))

    if malformed.height > 0:
        return {
            "status": "WARNING",
            "actual": {"malformed_count": malformed.height, "sample_ids": malformed["passengerid"].head(10).to_list()},
            "expected": "'Last, Title. First' pattern (contains a comma)",
            "message": f"{dataset_name}: {malformed.height} nama tidak mengikuti pola standar.",
        }

    return {
        "status": "PASS",
        "actual": 0,
        "expected": 0,
        "message": f"{dataset_name}: all names follow the expected pattern.",
    }


run_rule("DQ-017", "name_format", "train", "WARNING", False, check_name_format, train, "train")
run_rule("DQ-017", "name_format", "test", "WARNING", False, check_name_format, test, "test")

✓ [DQ-017] name_format (train) -> PASS: train: all names follow the expected pattern.
✓ [DQ-017] name_format (test) -> PASS: test: all names follow the expected pattern.


{'rule_id': 'DQ-017',
 'check': 'name_format',
 'dataset': 'test',
 'status': 'PASS',
 'severity': 'WARNING',
 'actual': '0',
 'expected': '0',
 'message': 'test: all names follow the expected pattern.'}

### DQ-018 — Numeric Domain
Sama seperti string quality, seluruh kolom numerik/kategorikal diperiksa
dalam satu ekspresi `.select()` per dataset.

In [29]:
def check_numeric_domains(df: pl.DataFrame, dataset_name: str) -> dict:
    columns = [c for c in NUMERIC_BOUNDS if c in df.columns]

    if not columns:
        return {"status": "PASS", "actual": {}, "expected": "n/a", "message": "No numeric columns to check."}

    exprs = []
    for c in columns:
        bounds = NUMERIC_BOUNDS[c]
        condition = pl.col(c).is_not_null()
        sub_conditions = []

        if bounds.get("min") is not None:
            sub_conditions.append(pl.col(c) < bounds["min"])
        if bounds.get("max") is not None:
            sub_conditions.append(pl.col(c) > bounds["max"])

        if not sub_conditions:
            continue

        invalid_condition = sub_conditions[0]
        for sc in sub_conditions[1:]:
            invalid_condition = invalid_condition | sc

        exprs.append((condition & invalid_condition).sum().alias(c))

    if not exprs:
        return {"status": "PASS", "actual": {}, "expected": "n/a", "message": "No bounded numeric columns to check."}

    stats = df.select(exprs).to_dicts()[0]
    violations = {c: int(n) for c, n in stats.items() if n > 0}

    if violations:
        return {
            "status": "FAIL",
            "actual": violations,
            "expected": "0 out-of-bound values per column",
            "message": f"{dataset_name}: kolom di luar domain numerik: {violations}",
        }

    return {
        "status": "PASS",
        "actual": {c: 0 for c in columns},
        "expected": "0 out-of-bound values",
        "message": f"{dataset_name}: numeric domain validation passed.",
    }


run_rule("DQ-018", "numeric_domain", "train", "CRITICAL", False, check_numeric_domains, train, "train")
run_rule("DQ-018", "numeric_domain", "test", "CRITICAL", False, check_numeric_domains, test, "test")


✓ [DQ-018] numeric_domain (train) -> PASS: train: numeric domain validation passed.
✓ [DQ-018] numeric_domain (test) -> PASS: test: numeric domain validation passed.


{'rule_id': 'DQ-018',
 'check': 'numeric_domain',
 'dataset': 'test',
 'status': 'PASS',
 'severity': 'CRITICAL',
 'actual': '{"age": 0, "fare": 0, "parch": 0, "sibsp": 0}',
 'expected': '0 out-of-bound values',
 'message': 'test: numeric domain validation passed.'}

### DQ-019 — Categorical Domain *(non-blocking)*

In [30]:
def check_categorical_domains(df: pl.DataFrame, dataset_name: str) -> dict:
    columns = [c for c in ALLOWED_VALUES if c in df.columns]

    if not columns:
        return {"status": "PASS", "actual": {}, "expected": "n/a", "message": "No categorical columns to check."}

    exprs = [
        (pl.col(c).is_not_null() & ~pl.col(c).is_in(list(ALLOWED_VALUES[c]))).sum().alias(c)
        for c in columns
    ]

    stats = df.select(exprs).to_dicts()[0]
    violating_columns = [c for c, n in stats.items() if n > 0]

    if violating_columns:
        details = {}
        for c in violating_columns:
            observed = set(df[c].drop_nulls().unique().to_list())
            details[c] = sorted(observed - ALLOWED_VALUES[c])

        return {
            "status": "FAIL",
            "actual": details,
            "expected": {c: sorted(ALLOWED_VALUES[c]) for c in columns},
            "message": f"{dataset_name}: nilai invalid ditemukan pada: {list(details.keys())}",
        }

    return {
        "status": "PASS",
        "actual": {c: sorted(ALLOWED_VALUES[c]) for c in columns},
        "expected": {c: sorted(ALLOWED_VALUES[c]) for c in columns},
        "message": f"{dataset_name}: categorical domain validation passed.",
    }


run_rule("DQ-019", "categorical_domain", "train", "CRITICAL", False, check_categorical_domains, train, "train")
run_rule("DQ-019", "categorical_domain", "test", "CRITICAL", False, check_categorical_domains, test, "test")

✓ [DQ-019] categorical_domain (train) -> PASS: train: categorical domain validation passed.
✓ [DQ-019] categorical_domain (test) -> PASS: test: categorical domain validation passed.


{'rule_id': 'DQ-019',
 'check': 'categorical_domain',
 'dataset': 'test',
 'status': 'PASS',
 'severity': 'CRITICAL',
 'actual': '{"embarked": ["C", "Q", "S"], "pclass": [1, 2, 3], "sex": ["female", "male"]}',
 'expected': '{"embarked": ["C", "Q", "S"], "pclass": [1, 2, 3], "sex": ["female", "male"]}',
 'message': 'test: categorical domain validation passed.'}

### DQ-020 — Train / Test Relationship *(non-blocking)*

In [31]:
def check_train_test_relationship(train_df: pl.DataFrame, test_df: pl.DataFrame) -> dict:
    train_ids = set(train_df["passengerid"].to_list())
    test_ids = set(test_df["passengerid"].to_list())
    overlap_ids = train_ids & test_ids

    if overlap_ids:
        return {
            "status": "FAIL",
            "actual": {"overlap_count": len(overlap_ids), "sample": sorted(overlap_ids)[:10]},
            "expected": {"overlap_count": 0},
            "message": f"Train/Test passengerid overlap: {len(overlap_ids)} IDs",
        }

    return {
        "status": "PASS",
        "actual": {"train_ids": len(train_ids), "test_ids": len(test_ids), "overlap_count": 0},
        "expected": {"overlap_count": 0},
        "message": "Train and test passenger IDs do not overlap.",
    }


relationship_result = run_rule(
    "DQ-020", "train_test_relationship", "all", "CRITICAL", False,
    check_train_test_relationship, train, test,
)

_rel_actual = json.loads(relationship_result["actual"])
overlap_id_count = _rel_actual.get("overlap_count", 0)

print("Train IDs:", train.height)
print("Test IDs :", test.height)
print("Overlap  :", overlap_id_count)


✓ [DQ-020] train_test_relationship (all) -> PASS: Train and test passenger IDs do not overlap.
Train IDs: 891
Test IDs : 418
Overlap  : 0


### Raw Dataset Summary, Preview, Statistics, Cardinality (observability only)

In [32]:
def dataset_summary(df: pl.DataFrame, dataset_name: str) -> dict:
    return {
        "dataset": dataset_name,
        "rows": df.height,
        "columns": df.width,
        "column_names": df.columns,
        "schema": {column: str(dtype) for column, dtype in df.schema.items()},
        "schema_sha256": schema_fingerprint(df),
        "estimated_size_bytes": df.estimated_size(),
    }


train_summary = dataset_summary(train, "train")
test_summary = dataset_summary(test, "test")

print(json.dumps(train_summary, indent=2, ensure_ascii=False))
print()
print(json.dumps(test_summary, indent=2, ensure_ascii=False))


{
  "dataset": "train",
  "rows": 891,
  "columns": 12,
  "column_names": [
    "passengerid",
    "survived",
    "pclass",
    "name",
    "sex",
    "age",
    "sibsp",
    "parch",
    "ticket",
    "fare",
    "cabin",
    "embarked"
  ],
  "schema": {
    "passengerid": "Int64",
    "survived": "Int64",
    "pclass": "Int64",
    "name": "String",
    "sex": "String",
    "age": "Float64",
    "sibsp": "Int64",
    "parch": "Int64",
    "ticket": "String",
    "fare": "Float64",
    "cabin": "String",
    "embarked": "String"
  },
  "schema_sha256": "0f3eb72d2000aa4f1d222676c7802f36bb4a945c72879844bbf02df1ee8530d2",
  "estimated_size_bytes": 85871
}

{
  "dataset": "test",
  "rows": 418,
  "columns": 11,
  "column_names": [
    "passengerid",
    "pclass",
    "name",
    "sex",
    "age",
    "sibsp",
    "parch",
    "ticket",
    "fare",
    "cabin",
    "embarked"
  ],
  "schema": {
    "passengerid": "Int64",
    "pclass": "Int64",
    "name": "String",
    "sex": "String",


In [33]:
print("TRAIN")
display(train.head(5))

print("TEST")
display(test.head(5))


TRAIN


passengerid,survived,pclass,name,sex,age,sibsp,parch,ticket,fare,cabin,embarked
i64,i64,i64,str,str,f64,i64,i64,str,f64,str,str
1,0,3,"""Braund, Mr. Owen Harris""","""male""",22.0,1,0,"""A/5 21171""",7.25,null,"""S"""
2,1,1,"""Cumings, Mrs. John Bradley (Fl…","""female""",38.0,1,0,"""PC 17599""",71.2833,"""C85""","""C"""
3,1,3,"""Heikkinen, Miss. Laina""","""female""",26.0,0,0,"""STON/O2. 3101282""",7.925,null,"""S"""
4,1,1,"""Futrelle, Mrs. Jacques Heath (…","""female""",35.0,1,0,"""113803""",53.1,"""C123""","""S"""
5,0,3,"""Allen, Mr. William Henry""","""male""",35.0,0,0,"""373450""",8.05,null,"""S"""


TEST


passengerid,pclass,name,sex,age,sibsp,parch,ticket,fare,cabin,embarked
i64,i64,str,str,f64,i64,i64,str,f64,str,str
892,3,"""Kelly, Mr. James""","""male""",34.5,0,0,"""330911""",7.8292,null,"""Q"""
893,3,"""Wilkes, Mrs. James (Ellen Need…","""female""",47.0,1,0,"""363272""",7.0,null,"""S"""
894,2,"""Myles, Mr. Thomas Francis""","""male""",62.0,0,0,"""240276""",9.6875,null,"""Q"""
895,3,"""Wirz, Mr. Albert""","""male""",27.0,0,0,"""315154""",8.6625,null,"""S"""
896,3,"""Hirvonen, Mrs. Alexander (Helg…","""female""",22.0,1,1,"""3101298""",12.2875,null,"""S"""


In [34]:
print("TRAIN DESCRIBE")
display(train.describe())

print()

print("TEST DESCRIBE")
display(test.describe())


TRAIN DESCRIBE


statistic,passengerid,survived,pclass,name,sex,age,sibsp,parch,ticket,fare,cabin,embarked
str,f64,f64,f64,str,str,f64,f64,f64,str,f64,str,str
"""count""",891.0,891.0,891.0,"""891""","""891""",714.0,891.0,891.0,"""891""",891.0,"""204""","""889"""
"""null_count""",0.0,0.0,0.0,"""0""","""0""",177.0,0.0,0.0,"""0""",0.0,"""687""","""2"""
"""mean""",446.0,0.383838,2.308642,null,null,29.699118,0.523008,0.381594,null,32.204208,null,null
"""std""",257.353842,0.486592,0.836071,null,null,14.526497,1.102743,0.806057,null,49.693429,null,null
"""min""",1.0,0.0,1.0,"""Abbing, Mr. Anthony""","""female""",0.42,0.0,0.0,"""110152""",0.0,"""A10""","""C"""
"""25%""",224.0,0.0,2.0,null,null,20.0,0.0,0.0,null,7.925,null,null
"""50%""",446.0,0.0,3.0,null,null,28.0,0.0,0.0,null,14.4542,null,null
"""75%""",669.0,1.0,3.0,null,null,38.0,1.0,0.0,null,31.0,null,null
"""max""",891.0,1.0,3.0,"""van Melkebeke, Mr. Philemon""","""male""",80.0,8.0,6.0,"""WE/P 5735""",512.3292,"""T""","""S"""



TEST DESCRIBE


statistic,passengerid,pclass,name,sex,age,sibsp,parch,ticket,fare,cabin,embarked
str,f64,f64,str,str,f64,f64,f64,str,f64,str,str
"""count""",418.0,418.0,"""418""","""418""",332.0,418.0,418.0,"""418""",417.0,"""91""","""418"""
"""null_count""",0.0,0.0,"""0""","""0""",86.0,0.0,0.0,"""0""",1.0,"""327""","""0"""
"""mean""",1100.5,2.26555,null,null,30.27259,0.447368,0.392344,null,35.627188,null,null
"""std""",120.810458,0.841838,null,null,14.181209,0.89676,0.981429,null,55.907576,null,null
"""min""",892.0,1.0,"""Abbott, Master. Eugene Joseph""","""female""",0.17,0.0,0.0,"""110469""",0.0,"""A11""","""C"""
"""25%""",996.0,1.0,null,null,21.0,0.0,0.0,null,7.8958,null,null
"""50%""",1101.0,3.0,null,null,27.0,0.0,0.0,null,14.4542,null,null
"""75%""",1205.0,3.0,null,null,39.0,1.0,0.0,null,31.5,null,null
"""max""",1309.0,3.0,"""van Billiard, Master. Walter J…","""male""",76.0,8.0,9.0,"""W.E.P. 5734""",512.3292,"""G6""","""S"""


In [35]:
def cardinality_report(df: pl.DataFrame) -> pl.DataFrame:
    exprs = []
    for column in df.columns:
        exprs.append(pl.col(column).n_unique().alias(f"{column}__unique"))
        exprs.append(pl.col(column).null_count().alias(f"{column}__null"))

    stats = df.select(exprs).to_dicts()[0]

    rows = [
        {
            "column": column,
            "unique_count": stats[f"{column}__unique"],
            "null_count": stats[f"{column}__null"],
        }
        for column in df.columns
    ]

    return pl.DataFrame(rows)


train_cardinality = cardinality_report(train)
test_cardinality = cardinality_report(test)

print("TRAIN CARDINALITY")
display(train_cardinality)

print()

print("TEST CARDINALITY")
display(test_cardinality)


TRAIN CARDINALITY


column,unique_count,null_count
str,i64,i64
"""passengerid""",891,0
"""survived""",2,0
"""pclass""",3,0
"""name""",891,0
"""sex""",2,0
"""age""",89,177
"""sibsp""",7,0
"""parch""",7,0
"""ticket""",681,0



TEST CARDINALITY


column,unique_count,null_count
str,i64,i64
"""passengerid""",418,0
"""pclass""",3,0
"""name""",418,0
"""sex""",2,0
"""age""",80,86
"""sibsp""",7,0
"""parch""",8,0
"""ticket""",363,0
"""fare""",170,1


### DQ-021 — Write Staged Artifacts *(non-blocking)*

In [36]:
def check_write_staged_artifacts() -> dict:
    train.write_parquet(TRAIN_PARQUET)
    test.write_parquet(TEST_PARQUET)

    if not TRAIN_PARQUET.exists() or not TEST_PARQUET.exists():
        return {
            "status": "FAIL",
            "actual": {"train_exists": TRAIN_PARQUET.exists(), "test_exists": TEST_PARQUET.exists()},
            "expected": "both parquet files exist",
            "message": "Staged Parquet artifacts gagal dibuat.",
        }

    if TRAIN_PARQUET.stat().st_size <= 0 or TEST_PARQUET.stat().st_size <= 0:
        return {
            "status": "FAIL",
            "actual": {"train_size": TRAIN_PARQUET.stat().st_size, "test_size": TEST_PARQUET.stat().st_size},
            "expected": "> 0 bytes",
            "message": "Staged Parquet artifacts kosong.",
        }

    return {
        "status": "PASS",
        "actual": {"train_size": TRAIN_PARQUET.stat().st_size, "test_size": TEST_PARQUET.stat().st_size},
        "expected": "> 0 bytes",
        "message": "Staged Parquet artifacts created successfully.",
    }


run_rule("DQ-021", "staged_write", "all", "CRITICAL", False, check_write_staged_artifacts)


✓ [DQ-021] staged_write (all) -> PASS: Staged Parquet artifacts created successfully.


{'rule_id': 'DQ-021',
 'check': 'staged_write',
 'dataset': 'all',
 'status': 'PASS',
 'severity': 'CRITICAL',
 'actual': '{"test_size": 15532, "train_size": 27613}',
 'expected': '> 0 bytes',
 'message': 'Staged Parquet artifacts created successfully.'}

### DQ-022 — Staged Artifact Read-Back

In [37]:
train_staged_sha256 = sha256_file(TRAIN_PARQUET) if TRAIN_PARQUET.exists() else None
test_staged_sha256 = sha256_file(TEST_PARQUET) if TEST_PARQUET.exists() else None

train_staged_size = TRAIN_PARQUET.stat().st_size if TRAIN_PARQUET.exists() else 0
test_staged_size = TEST_PARQUET.stat().st_size if TEST_PARQUET.exists() else 0

print("Train artifact SHA256:", train_staged_sha256)
print("Test  artifact SHA256:", test_staged_sha256)


def check_staged_readback() -> dict:
    global train_staged, test_staged

    train_staged = pl.read_parquet(TRAIN_PARQUET)
    test_staged = pl.read_parquet(TEST_PARQUET)

    issues = []

    if train_staged.columns != EXPECTED_TRAIN_COLUMNS:
        issues.append("train columns mismatch")
    if test_staged.columns != EXPECTED_TEST_COLUMNS:
        issues.append("test columns mismatch")
    if train_staged.height != train.height:
        issues.append("train row count mismatch")
    if test_staged.height != test.height:
        issues.append("test row count mismatch")
    if train_staged.width != train.width:
        issues.append("train column count mismatch")
    if test_staged.width != test.width:
        issues.append("test column count mismatch")

    for dtype_check, df_, name_ in ((check_dtypes, train_staged, "train_staged"), (check_dtypes, test_staged, "test_staged")):
        dtype_result = dtype_check(df_, name_)
        if dtype_result["status"] != "PASS":
            issues.append(dtype_result["message"])

    if issues:
        return {
            "status": "FAIL",
            "actual": issues,
            "expected": "staged artifact structurally identical to in-memory dataframe",
            "message": "; ".join(issues),
        }

    return {
        "status": "PASS",
        "actual": {
            "train": {"rows": train_staged.height, "columns": train_staged.width},
            "test": {"rows": test_staged.height, "columns": test_staged.width},
        },
        "expected": "readable and structurally identical staged artifacts",
        "message": "Staged artifacts readable and structurally valid.",
    }


run_rule("DQ-022", "staged_readback", "all", "CRITICAL", False, check_staged_readback)


Train artifact SHA256: 8057e012ff35427402e0e22e1227e691a2afa78e99595ed7c5c8b5bc9110c271
Test  artifact SHA256: aab5bc44778f5c903fa35786c16a4849533b4a5afff20e031575745a4eb3950c
✓ [DQ-022] staged_readback (all) -> PASS: Staged artifacts readable and structurally valid.


{'rule_id': 'DQ-022',
 'check': 'staged_readback',
 'dataset': 'all',
 'status': 'PASS',
 'severity': 'CRITICAL',
 'actual': '{"test": {"columns": 11, "rows": 418}, "train": {"columns": 12, "rows": 891}}',
 'expected': 'readable and structurally identical staged artifacts',
 'message': 'Staged artifacts readable and structurally valid.'}

### DQ-023 — Staged Schema Fingerprint Match *(non-blocking)*

In [38]:
def check_staged_schema_fingerprint() -> dict:
    train_staged_schema_sha = schema_fingerprint(train_staged)
    test_staged_schema_sha = schema_fingerprint(test_staged)

    matches = (train_staged_schema_sha == train_schema_sha256) and (test_staged_schema_sha == test_schema_sha256)

    if not matches:
        return {
            "status": "FAIL",
            "actual": {"train_staged": train_staged_schema_sha, "test_staged": test_staged_schema_sha},
            "expected": {"train": train_schema_sha256, "test": test_schema_sha256},
            "message": "Staged schema fingerprint tidak cocok dengan source schema.",
        }

    return {
        "status": "PASS",
        "actual": {"train_staged": train_staged_schema_sha, "test_staged": test_staged_schema_sha},
        "expected": {"train": train_schema_sha256, "test": test_schema_sha256},
        "message": "Staged schema fingerprints match source schema.",
    }


staged_fp_result = run_rule("DQ-023", "staged_schema_fingerprint", "all", "CRITICAL", False, check_staged_schema_fingerprint)
_staged_fp_actual = json.loads(staged_fp_result["actual"])
train_staged_schema_sha256 = _staged_fp_actual["train_staged"]
test_staged_schema_sha256 = _staged_fp_actual["test_staged"]

✓ [DQ-023] staged_schema_fingerprint (all) -> PASS: Staged schema fingerprints match source schema.


## Validation Summary & Severity-Aware Quality Gate

Perbaikan kunci di sini: `overall_status` **tidak lagi** dihitung dari
"apakah ada status FAIL di mana pun" (yang membuat `severity` jadi tidak berguna),
melainkan:

- `FAIL` → jika ada rule dengan **severity CRITICAL** berstatus `FAIL`/`ERROR`.
- `PASS_WITH_WARNINGS` → jika tidak ada kegagalan CRITICAL, tetapi ada rule
  (severity apa pun) berstatus `WARNING`, atau rule *severity* WARNING yang gagal.
- `PASS` → selain itu.

In [39]:
validation_df = pl.DataFrame(
    validation_results,
    schema={
        "rule_id": pl.String,
        "check": pl.String,
        "dataset": pl.String,
        "status": pl.String,
        "severity": pl.String,
        "actual": pl.String,
        "expected": pl.String,
        "message": pl.String,
    },
)

display(validation_df)


rule_id,check,dataset,status,severity,actual,expected,message
str,str,str,str,str,str,str,str
"""DQ-001""","""source_files""","""all""","""PASS""","""CRITICAL""","""{""test"": {""extension"": "".csv"",…","""existing, non-empty .csv files""","""Source files exist and are val…"
"""DQ-002""","""source_fingerprint""","""all""","""PASS""","""CRITICAL""","""{""test_sha256"": ""56023b9948236…","""valid SHA-256 fingerprint""","""Source SHA-256 fingerprints ge…"
"""DQ-003""","""csv_structure""","""train""","""PASS""","""CRITICAL""","""{""delimiter"": "","", ""encoding"":…","""utf-8 CSV, comma-delimited, un…","""CSV structural validation pass…"
"""DQ-003""","""csv_structure""","""test""","""PASS""","""CRITICAL""","""{""delimiter"": "","", ""encoding"":…","""utf-8 CSV, comma-delimited, un…","""CSV structural validation pass…"
"""DQ-004""","""raw_load""","""all""","""PASS""","""CRITICAL""","""{""test_shape"": [418, 11], ""tra…","""non-empty dataframes""","""Raw CSV loaded successfully."""
"""DQ-005""","""column_normalization""","""all""","""PASS""","""CRITICAL""","""{""test_columns"": [""passengerid…","""unique lowercase column names""","""Column names normalized to low…"
"""DQ-006""","""column_schema""","""train""","""PASS""","""CRITICAL""","""[""passengerid"", ""survived"", ""p…","""[""passengerid"", ""survived"", ""p…","""train column schema matches th…"
"""DQ-007""","""column_schema""","""test""","""PASS""","""CRITICAL""","""[""passengerid"", ""pclass"", ""nam…","""[""passengerid"", ""pclass"", ""nam…","""test column schema matches the…"
"""DQ-008""","""data_types""","""train""","""PASS""","""CRITICAL""","""{""age"": ""Float64"", ""cabin"": ""S…","""{""age"": ""Float64"", ""cabin"": ""S…","""train: data types match the sc…"


In [40]:
# ============================================================
# SEVERITY-AWARE OVERALL QUALITY GATE
# ============================================================

critical_failures = validation_df.filter(
    (pl.col("severity") == "CRITICAL") & (pl.col("status").is_in(["FAIL", "ERROR"]))
)

any_warning_signal = validation_df.filter(
    (pl.col("status") == "WARNING")
    | ((pl.col("severity") == "WARNING") & (pl.col("status").is_in(["FAIL", "ERROR"])))
)

if critical_failures.height > 0:
    overall_status = "FAIL"
elif any_warning_signal.height > 0:
    overall_status = "PASS_WITH_WARNINGS"
else:
    overall_status = "PASS"

print("=" * 72)
print("OVERALL INGESTION QUALITY GATE")
print("=" * 72)
print(overall_status)

if critical_failures.height > 0:
    print()
    print("CRITICAL FAILURES:")
    display(critical_failures)

if any_warning_signal.height > 0:
    print()
    print("WARNING SIGNALS:")
    display(any_warning_signal)

print("=" * 72)


OVERALL INGESTION QUALITY GATE
PASS_WITH_WARNINGS

WARNING SIGNALS:


rule_id,check,dataset,status,severity,actual,expected,message
str,str,str,str,str,str,str,str
"""DQ-016""","""string_quality""","""train""","""WARNING""","""WARNING""","""{""cabin"": {""empty_count"": 0, ""…","""{""empty_strings"": 0, ""leading_…","""train: string quality checked …"
"""DQ-016""","""string_quality""","""test""","""WARNING""","""WARNING""","""{""cabin"": {""empty_count"": 0, ""…","""{""empty_strings"": 0, ""leading_…","""test: string quality checked w…"


In [41]:
# ============================================================
# VALIDATION STATISTICS
# ============================================================

validation_counts = {
    "total_rules": validation_df.height,
    "passed": validation_df.filter(pl.col("status") == "PASS").height,
    "warnings": validation_df.filter(pl.col("status") == "WARNING").height,
    "failed": validation_df.filter(pl.col("status") == "FAIL").height,
    "errored": validation_df.filter(pl.col("status") == "ERROR").height,
    "critical_failures": critical_failures.height,
}

print(json.dumps(validation_counts, indent=2))


{
  "total_rules": 36,
  "passed": 34,
  "warnings": 2,
  "failed": 0,
  "errored": 0,
  "critical_failures": 0
}


### Provenance & Environment

In [42]:
def get_git_info(timeout_seconds: float = 5.0) -> dict:
    result = {"repository": None, "branch": None, "commit": None, "available": False}

    try:
        repository = subprocess.run(
            ["git", "config", "--get", "remote.origin.url"],
            capture_output=True, text=True, check=False, timeout=timeout_seconds,
        )
        branch = subprocess.run(
            ["git", "branch", "--show-current"],
            capture_output=True, text=True, check=False, timeout=timeout_seconds,
        )
        commit = subprocess.run(
            ["git", "rev-parse", "HEAD"],
            capture_output=True, text=True, check=False, timeout=timeout_seconds,
        )

        if commit.returncode == 0:
            result["available"] = True
            result["repository"] = repository.stdout.strip() if repository.returncode == 0 else None
            result["branch"] = branch.stdout.strip() if branch.returncode == 0 else None
            result["commit"] = commit.stdout.strip()

    except (FileNotFoundError, subprocess.TimeoutExpired):
        pass

    return result


git_info = get_git_info()

print(json.dumps(git_info, indent=2, ensure_ascii=False))


{
  "repository": "https://github.com/prasetya-code/titanic_survival_analysis.git",
  "branch": "main",
  "commit": "97d2acc7b9635b5de81c0882eefe47d748e8082a",
  "available": true
}


In [43]:
MIN_PYTHON_VERSION = (3, 10)
RECOMMENDED_POLARS_VERSION = "1.0.0"

environment_info = {
    "python": sys.version,
    "python_executable": sys.executable,
    "platform": platform.platform(),
    "machine": platform.machine(),
    "processor": platform.processor(),
    "polars": pl.__version__,
}

python_version_ok = sys.version_info[:2] >= MIN_PYTHON_VERSION

if not python_version_ok:
    print(
        f"⚠ Python {sys.version_info[:2]} lebih rendah dari minimum yang "
        f"direkomendasikan {MIN_PYTHON_VERSION}. Reproducibility dapat terpengaruh."
    )

print(json.dumps(environment_info, indent=2, ensure_ascii=False))


{
  "python": "3.12.0 (tags/v3.12.0:0fb18b0, Oct  2 2023, 13:03:39) [MSC v.1935 64 bit (AMD64)]",
  "python_executable": "d:\\GIT DATA\\titanic_survival_analysis\\win_env\\Scripts\\python.exe",
  "platform": "Windows-11-10.0.26200-SP0",
  "machine": "AMD64",
  "processor": "Intel64 Family 6 Model 126 Stepping 5, GenuineIntel",
  "polars": "1.44.2"
}


In [44]:
INGESTION_END = time.perf_counter()

INGESTION_DURATION_SECONDS = round(INGESTION_END - INGESTION_START, 4)

print(f"Ingestion duration: {INGESTION_DURATION_SECONDS} seconds")


Ingestion duration: 2.2713 seconds


### Ingestion Metadata & Validation Report

Kedua artifact ditulis **dua kali**: sekali sebagai file "latest" (nama tetap,
untuk dikonsumsi tahap berikutnya di pipeline) dan sekali sebagai arsip
ber-`RUN_ID` di `metadata/history/` (untuk audit trail historis antar-run).

In [45]:
ingestion_timestamp_utc = RUN_TIMESTAMP.isoformat()

metadata = {
    "run": {
        "run_id": RUN_ID,
        "started_at_utc": ingestion_timestamp_utc,
        "duration_seconds": INGESTION_DURATION_SECONDS,
    },
    "dataset": {
        "name": DATASET_NAME,
        "source_format": SOURCE_FORMAT,
        "artifact_format": ARTIFACT_FORMAT,
    },
    "pipeline": {
        "stage": PIPELINE_STAGE,
        "pipeline_version": PIPELINE_VERSION,
        "schema_version": SCHEMA_VERSION,
        "dq_rules_version": DQ_RULES_VERSION,
    },
    "source": {
        "train": {
            "path": str(TRAIN_RAW),
            "file_name": TRAIN_RAW.name,
            "size_bytes": train_file_info["size_bytes"],
            "sha256": train_sha256,
            "format": SOURCE_FORMAT,
            "encoding": train_csv_structure["encoding"],
            "delimiter": train_csv_structure["delimiter"],
        },
        "test": {
            "path": str(TEST_RAW),
            "file_name": TEST_RAW.name,
            "size_bytes": test_file_info["size_bytes"],
            "sha256": test_sha256,
            "format": SOURCE_FORMAT,
            "encoding": test_csv_structure["encoding"],
            "delimiter": test_csv_structure["delimiter"],
        },
    },
    "schema": {
        "train": {"columns": EXPECTED_TRAIN_COLUMNS, "fingerprint_sha256": train_schema_sha256},
        "test": {"columns": EXPECTED_TEST_COLUMNS, "fingerprint_sha256": test_schema_sha256},
    },
    "datasets": {"train": train_summary, "test": test_summary},
    "artifacts": {
        "train": {
            "path": str(TRAIN_PARQUET), "format": ARTIFACT_FORMAT,
            "size_bytes": train_staged_size, "sha256": train_staged_sha256,
            "schema_sha256": train_staged_schema_sha256,
            "rows": train_staged.height, "columns": train_staged.width,
        },
        "test": {
            "path": str(TEST_PARQUET), "format": ARTIFACT_FORMAT,
            "size_bytes": test_staged_size, "sha256": test_staged_sha256,
            "schema_sha256": test_staged_schema_sha256,
            "rows": test_staged.height, "columns": test_staged.width,
        },
    },
    "validation": {
        "overall_status": overall_status,
        "statistics": validation_counts,
        "rules": validation_results,
    },
    "metrics": {
        "train_rows": train.height,
        "test_rows": test.height,
        "train_columns": train.width,
        "test_columns": test.width,
        "train_full_row_duplicates": train_duplicate_rows,
        "test_full_row_duplicates": test_duplicate_rows,
        "train_test_id_overlap": overlap_id_count,
    },
    "provenance": {"git": git_info},
    "environment": environment_info,
    "status": overall_status,
}

INGESTION_METADATA.write_text(json.dumps(metadata, indent=2, ensure_ascii=False), encoding="utf-8")
(HISTORY_DIR / f"{RUN_ID}_ingestion_metadata.json").write_text(
    json.dumps(metadata, indent=2, ensure_ascii=False), encoding="utf-8"
)

print("✓ Metadata written:", INGESTION_METADATA)
print("✓ Metadata archived:", HISTORY_DIR / f"{RUN_ID}_ingestion_metadata.json")


✓ Metadata written: d:\GIT DATA\titanic_survival_analysis\data\stage\metadata\ingestion_metadata.json
✓ Metadata archived: d:\GIT DATA\titanic_survival_analysis\data\stage\metadata\history\titanic-ingestion-20260922T032421Z_ingestion_metadata.json


In [46]:
validation_report = {
    "run_id": RUN_ID,
    "dataset": DATASET_NAME,
    "pipeline_stage": PIPELINE_STAGE,
    "pipeline_version": PIPELINE_VERSION,
    "schema_version": SCHEMA_VERSION,
    "dq_rules_version": DQ_RULES_VERSION,
    "timestamp_utc": ingestion_timestamp_utc,
    "overall_status": overall_status,
    "statistics": validation_counts,
    "rules": validation_results,
}

VALIDATION_REPORT.write_text(json.dumps(validation_report, indent=2, ensure_ascii=False), encoding="utf-8")
(HISTORY_DIR / f"{RUN_ID}_validation_report.json").write_text(
    json.dumps(validation_report, indent=2, ensure_ascii=False), encoding="utf-8"
)

print("✓ Validation report written:", VALIDATION_REPORT)
print("✓ Validation report archived:", HISTORY_DIR / f"{RUN_ID}_validation_report.json")


✓ Validation report written: d:\GIT DATA\titanic_survival_analysis\data\stage\metadata\validation_report.json
✓ Validation report archived: d:\GIT DATA\titanic_survival_analysis\data\stage\metadata\history\titanic-ingestion-20260922T032421Z_validation_report.json


In [47]:
def build_markdown_report() -> str:
    lines = []
    lines.append(f"# Ingestion Validation Report — `{RUN_ID}`")
    lines.append("")
    lines.append(f"- **Dataset**: {DATASET_NAME}")
    lines.append(f"- **Timestamp (UTC)**: {ingestion_timestamp_utc}")
    lines.append(f"- **Overall status**: **{overall_status}**")
    lines.append(f"- **Duration**: {INGESTION_DURATION_SECONDS}s")
    lines.append("")
    lines.append(
        f"| Total | Passed | Warnings | Failed | Errored |\n"
        f"|---|---|---|---|---|\n"
        f"| {validation_counts['total_rules']} | {validation_counts['passed']} | "
        f"{validation_counts['warnings']} | {validation_counts['failed']} | "
        f"{validation_counts['errored']} |"
    )
    lines.append("")
    lines.append("## Rule Detail")
    lines.append("")
    lines.append("| Rule | Check | Dataset | Severity | Status | Message |")
    lines.append("|---|---|---|---|---|---|")

    for r in validation_results:
        message = r["message"].replace("|", "\\|")
        lines.append(
            f"| {r['rule_id']} | {r['check']} | {r['dataset']} | {r['severity']} | "
            f"{r['status']} | {message} |"
        )

    return "\n".join(lines) + "\n"


VALIDATION_REPORT_MD.write_text(build_markdown_report(), encoding="utf-8")

print("✓ Human-readable Markdown report written:", VALIDATION_REPORT_MD)


✓ Human-readable Markdown report written: d:\GIT DATA\titanic_survival_analysis\data\stage\metadata\validation_report.md


### Final Ingestion Report

In [48]:
final_report = {
    "run_id": RUN_ID,
    "status": overall_status,
    "dataset": DATASET_NAME,
    "train": {
        "rows": train.height, "columns": train.width,
        "source_sha256": train_sha256, "artifact_sha256": train_staged_sha256,
    },
    "test": {
        "rows": test.height, "columns": test.width,
        "source_sha256": test_sha256, "artifact_sha256": test_staged_sha256,
    },
    "validation": validation_counts,
    "duration_seconds": INGESTION_DURATION_SECONDS,
    "metadata_path": str(INGESTION_METADATA),
    "validation_report_path": str(VALIDATION_REPORT),
    "validation_report_markdown_path": str(VALIDATION_REPORT_MD),
}

print("=" * 72)
print("FINAL INGESTION REPORT")
print("=" * 72)
print(json.dumps(final_report, indent=2, ensure_ascii=False))
print("=" * 72)
print("INGESTION QUALITY GATE:", overall_status)
print("=" * 72)


FINAL INGESTION REPORT
{
  "run_id": "titanic-ingestion-20260922T032421Z",
  "status": "PASS_WITH_WARNINGS",
  "dataset": "titanic",
  "train": {
    "rows": 891,
    "columns": 12,
    "source_sha256": "7d118fef8b6ccf7f81111877bc388536f7b1e498a655e3d649d19aaa010e9f6f",
    "artifact_sha256": "8057e012ff35427402e0e22e1227e691a2afa78e99595ed7c5c8b5bc9110c271"
  },
  "test": {
    "rows": 418,
    "columns": 11,
    "source_sha256": "56023b9948236f3c7a1c9448fcf418b283e109ef177fa8c7e069158dd7dd52b2",
    "artifact_sha256": "aab5bc44778f5c903fa35786c16a4849533b4a5afff20e031575745a4eb3950c"
  },
  "validation": {
    "total_rules": 36,
    "passed": 34,
    "warnings": 2,
    "failed": 0,
    "errored": 0,
    "critical_failures": 0
  },
  "duration_seconds": 2.2713,
  "metadata_path": "d:\\GIT DATA\\titanic_survival_analysis\\data\\stage\\metadata\\ingestion_metadata.json",
  "validation_report_path": "d:\\GIT DATA\\titanic_survival_analysis\\data\\stage\\metadata\\validation_report.json",

### Final Quality Gate

Notebook baru saja berhasil sampai di titik ini walau mungkin ada rule yang
`FAIL` (karena rule non-blocking tetap dijalankan semua). Gate final di bawah
ini yang benar-benar memutuskan apakah pipeline **boleh lanjut** ke tahap
berikutnya — dan itu terjadi **setelah** seluruh laporan tertulis ke disk.

In [49]:
if overall_status == "FAIL":
    raise RuntimeError(
        "INGESTION FAILED (CRITICAL). "
        f"Periksa {VALIDATION_REPORT} atau {VALIDATION_REPORT_MD} untuk detail."
    )

print("✓ INGESTION QUALITY GATE PASSED")

if overall_status == "PASS_WITH_WARNINGS":
    print("⚠ Dataset dapat dilanjutkan ke tahap berikutnya dengan warning yang tercatat.")
else:
    print("✓ Tidak ada warning pada ingestion.")


✓ INGESTION QUALITY GATE PASSED
⚠ Dataset dapat dilanjutkan ke tahap berikutnya dengan warning yang tercatat.


### Resource Cleanup

In [50]:
# Jangan melakukan cleanup sebelum:
# - validation selesai
# - artifact selesai dibuat
# - metadata selesai dibuat
# - final report selesai dibuat

objects_to_delete = [
    "train_staged", "test_staged", "train", "test",
    "train_missing", "test_missing",
    "train_string_quality_result", "test_string_quality_result",
    "train_cardinality", "test_cardinality",
    "validation_df",
]

deleted_objects = []

for object_name in objects_to_delete:
    if object_name in globals():
        del globals()[object_name]
        deleted_objects.append(object_name)

collected_objects = gc.collect()

print("✓ Resource cleanup completed")
print("Deleted objects:", deleted_objects)
print("Garbage collected:", collected_objects)


✓ Resource cleanup completed
Deleted objects: ['train_staged', 'test_staged', 'train', 'test', 'train_missing', 'test_missing', 'train_string_quality_result', 'test_string_quality_result', 'train_cardinality', 'test_cardinality', 'validation_df']
Garbage collected: 175


In [51]:
print("=" * 72)
print("INGESTION PIPELINE COMPLETED")
print("=" * 72)
print("Run ID :", RUN_ID)
print()
print("Train artifact:")
print(TRAIN_PARQUET)
print()
print("Test artifact:")
print(TEST_PARQUET)
print()
print("Metadata:")
print(INGESTION_METADATA)
print()
print("Validation report (JSON):")
print(VALIDATION_REPORT)
print()
print("Validation report (Markdown):")
print(VALIDATION_REPORT_MD)
print()
print("Final status:")
print(overall_status)
print("=" * 72)


INGESTION PIPELINE COMPLETED
Run ID : titanic-ingestion-20260922T032421Z

Train artifact:
d:\GIT DATA\titanic_survival_analysis\data\stage\train.parquet

Test artifact:
d:\GIT DATA\titanic_survival_analysis\data\stage\test.parquet

Metadata:
d:\GIT DATA\titanic_survival_analysis\data\stage\metadata\ingestion_metadata.json

Validation report (JSON):
d:\GIT DATA\titanic_survival_analysis\data\stage\metadata\validation_report.json

Validation report (Markdown):
d:\GIT DATA\titanic_survival_analysis\data\stage\metadata\validation_report.md

Final status:
PASS_WITH_WARNINGS
